<span style="color:#1a5276">**实验记录 — Two-Stage 肝脏/肿瘤分割**</span>

**任务:** MSD Task03 Liver，两阶段分割（Stage1 肝脏 → Stage2 肿瘤） | **模型:** DynUNet (MONAI)

- train: 92 cases（有肿瘤 82，无肿瘤 10）｜ val: 26 cases ｜ test: 13 cases（seed=0）

---
<span style="color:#1a5276">**Stage 1 — 肝脏模型（所有实验共用，不再重训）**</span>

| 项目 | 值 |
|------|----|
| 目录 | `dynunet_liver_only/train/03-14-01-11-56` |
| 模型 | DynUNet |
| Optimizer | SGD, lr=3e-3, momentum=0.99, weight_decay=3e-5 |
| LR Scheduler | Poly 0.9 |
| Loss | DiceCE |
| Patch | 144×144×144 |
| Epochs | 200 |
| batch_size | 1 |
| 数据备注 | label 1（肝脏）和 label 2（肿瘤）合并为 1 训练，train=92, val=26 |
| 训练时长 | 22.93h |
| GPU | NVIDIA RTX 2080 Ti |
| **Val liver Dice (best)** | **0.8905**（epoch 198） |
| **Test liver Dice** | **0.9594** |

---
<span style="color:#186a3b">**Stage 2 实验一 ✅ — 基础肿瘤模型**</span>

**目录:** `twostage/tumor_dynunet/train/03-17-09-22-42`

| 项目 | 值 |
|------|----|
| init_ckpt | Stage 1 best.pt（迁移学习） |
| Optimizer | AdamW, lr=3e-3 |
| Loss | DiceFocal |
| Patch | 96×96×96, batch_size=1 |
| Epochs | 300 |
| 数据 | train=82（仅肿瘤阳性 case），val=24 |
| bbox_jitter | ❌ 无 |
| random_margin | ❌ 无，固定 margin=8 |
| tumor_ratios | [0.05, 0.95] |
| repeats | 6 |
| 训练时长 | 29.02h |
| **Val tumor Dice (best)** | **0.3801**（epoch 264） |

**Eval 结果（test split，13 cases）**

| eval 时间 | liver Dice | tumor Dice | 备注 |
|-----------|-----------|-----------|------|
| 03-18-15-13 | 0.8927 | 0.5096 | liver Dice 偏低 |
| 03-18-16-29 | 0.9594 | **0.5677** | 本次最优 |
| 03-18-17-34 | 0.9594 | 0.5401 | — |

---
<span style="color:#186a3b">**Stage 2 实验二 ✅ — ROI Jitter 肿瘤模型（当前最强基线）**</span>

**commit:** `a95d1c3` | **目录:** `twostage/tumor_dynunet_roi_jitter/train/03-22-11-44-00`

核心改动：① 训练数据扩展到 92 cases ② bbox_jitter（max_shift=8）③ random_margin（[8,20]）

| 项目 | 值 |
|------|----|
| Optimizer | AdamW, lr=3e-3 |
| Loss | DiceFocal |
| Patch | 96×96×96, batch_size=2 |
| Epochs | 300 |
| bbox_jitter | ✅ max_shift=8 |
| random_margin | ✅ [8, 20] |
| 训练时长 | **45.92h** |
| **Val tumor Dice (best)** | **0.4697**（epoch 243） |

**训练曲线**

| epoch | loss | val tumor Dice | best |
|-------|------|---------------|------|
| 60 | 0.5158 | 0.1809 | 0.2478 |
| 120 | 0.4591 | 0.2958 | 0.3768 |
| 180 | 0.3439 | 0.3882 | 0.4277 |
| 240 | 0.3367 | 0.4294 | 0.4430 |
| 300 | 0.3111 | 0.4415 | **0.4697** |

**Eval 结果（test split，13 cases）**

| 指标 | mean | std | min | max |
|------|------|-----|-----|-----|
| liver Dice | 0.9594 | 0.026 | 0.879 | 0.976 |
| tumor Dice | **0.5816** | 0.288 | 0.023 | 1.000 |
| tumor Recall | 0.491 | 0.253 | 0.000 | 0.853 |
| tumor Precision | 0.631 | 0.377 | 0.000 | 0.989 |
| tumor FDR | 0.292 | 0.341 | 0.000 | 0.988 |
| tumor FNR | 0.432 | 0.243 | 0.000 | 0.923 |

**Per-case 明细**

| case | tumor Dice | Recall | FDR | 备注 |
|------|-----------|--------|-----|------|
| liver_7 | 0.860 | 0.853 | 0.132 | — |
| liver_87 | 1.000 | 0.000 | 0.000 | 无肿瘤 case，正确预测为空 |
| liver_27 | 0.805 | 0.828 | 0.216 | — |
| liver_71 | 0.478 | 0.315 | 0.011 | 大肿瘤，只找到 31% |
| liver_4 | 0.511 | 0.354 | 0.080 | 超大肿瘤，只找到 35% |
| liver_15 | 0.023 | 0.485 | 0.988 | 预测区域 99% 是错的 |
| liver_77 | 0.086 | 0.077 | 0.902 | 几乎全漏+严重误报 |

**问题：** FNR=0.432（主因，大肿瘤漏检） / FDR=0.292（次因，liver_15/77/107 严重误报）

<span style="color:#7d6608">**Eval-only: 实验二 + TTA ✅**</span>

**目录:** `eval_tta/03-22-11-44-00` | **commit:** `2014275` | 推理时长 0.633h（无 TTA 0.198h）

| 指标 | 无 TTA | TTA | Δ |
|------|--------|-----|---|
| liver Dice | 0.9594 | 0.9594 | = |
| tumor Dice | 0.5816 | **0.5965** ✅ | +0.015 |
| tumor Recall | 0.4914 | **0.5224** | +0.031 |
| tumor FNR | 0.4317 | **0.4007** | −0.031 |
| tumor FDR | 0.2920 | 0.2890 | ≈ |

TTA 主要提升 Recall（减少漏检），不引入额外假阳性。**当前 test split 最优：0.5965**

<span style="color:#7d6608">**Eval-only: 实验二 + TTA + min_tumor_size 扫参 ✅（val split，n=26）**</span>

`min_tumor_size` 过滤 Stage2 中体素数过少的预测区域，减少假阳性。⚠️ 使用 val split，不可与 test split 直接比较。

| min_tumor_size | Tumor Dice | Recall | Precision | FDR | FNR |
|---|---|---|---|---|---|
| 0（无过滤） | 0.5377 | 0.4315 | 0.7428 | 0.2188 | 0.4915 |
| **100 ← 最优** | **0.5757** | 0.4310 | 0.7450 | **0.1011** | 0.4921 |
| 200 | 0.5751 | 0.4301 | 0.7068 | 0.1009 | 0.4930 |
| 500 | 0.5463 | 0.4063 | 0.6715 | 0.0977 | 0.5167 |
| 1000 | 0.5079 | 0.3777 | 0.6019 | 0.0904 | 0.5454 |

0→100：FDR 大幅下降（0.219→0.101），Recall 几乎不变。200+：开始漏检真实肿瘤。**后续实验固定 `--min_tumor_size 100`**

<span style="color:#7d6608">**Eval-only: 实验二 + TTA + min_tumor_size 扫参 ✅（test split，n=13）**</span>

**目录:** `eval_min_tumor_size_sweep/` | 使用 `--tta --margin 12`，在 test split 上验证 val 扫参结论。

| min_tumor_size | Tumor Dice | Recall | Precision | FDR | FNR | Jaccard |
|---|---|---|---|---|---|---|
| 0（无过滤） | 0.5186 | 0.5231 | 0.6295 | 0.3705 | 0.4000 | 0.3993 |
| **100 ← 最优** | **0.5950** | 0.5187 | **0.6395** | 0.2836 | 0.4044 | 0.3989 |
| 200 | 0.5815 | 0.5052 | 0.6370 | 0.2861 | 0.4179 | 0.3883 |
| 500 | 0.5527 | 0.4304 | 0.5843 | 0.2618 | 0.4927 | 0.3711 |
| 1000 | 0.5442 | 0.4132 | 0.5854 | 0.2607 | 0.5099 | 0.3633 |

0→100：FDR 下降（0.371→0.284），Recall 基本不变。200+：Recall 明显下降，真实小肿瘤被误删。与 val split 结论一致，**`--min_tumor_size 100` 为最优，代码默认值已是 100，无需修改。**

---
<span style="color:#922b21">**Stage 2 实验三 ❌ — 困难样本 Fine-tune（失败）**</span>

**commit:** `994e6ee` | **目录:** `twostage/tumor_dynunet_hardfinetune/train/03-24-17-09-06`

| 项目 | 值 |
|------|----|
| init_ckpt | 实验二 best.pt |
| lr | 3e-4 |
| Epochs | 60（目标） |
| hard_cases_only | ✅ 小肿瘤（<5000 voxel）+ 无肿瘤 |
| 状态 | ⚠️ **15/60 epoch 中断** |
| best val tumor Dice | 0.4603（epoch 10）— 未超过实验二 0.4697 |

**失败原因：** lr=3e-4 在仅限困难样本的数据分布下收敛不稳定；仅 15 epoch 就中断，未给模型足够训练时间。

---
<span style="color:#922b21">**Stage 2 实验四 ❌ — pred_bbox ROI + Hard Mining（停止）**</span>

**目录:** `twostage/tumor_dynunet_predbbox_roi/train/03-27-11-11-47`

| 项目 | 值 |
|------|----|
| preprocessed_root | Task03_Liver_roi（tight_bbox = pred_bbox_stage1） |
| repeats | 6 + small_tumor_repeat_scale=4 + no_tumor_repeat_scale=2 |
| Epochs | 300 |
| 状态 | ❌ **手动停止（epoch 1 后）** |

**停止原因：** ~972 batches/epoch × 4.3s = 69 min/epoch → **300ep 约 345h（14天）**；同时 pred_bbox 和 hard mining 两变量混在一起，无法做消融对比。

---
<span style="color:#1a5276">**Stage 2 实验五 🏃 — pred_bbox ROI 干净对比（训练中）**</span>

**exp_name:** `tumor_dynunet_predbbox_roi_clean`

| 项目 | 值 |
|------|----|
| init_ckpt | 实验二 best.pt（03-22-11-44-00） |
| preprocessed_root | Task03_Liver_roi（tight_bbox = Stage1 pred_bbox） |
| Epochs | **300**（与实验二一致，控制变量） |
| repeats | **1，无 hard mining** → ~92 batches/epoch，~6.5 min/epoch |
| random_margin | ✅ [8, 24] |
| 预计时长 | **~33h** |
| 状态 | 🏃 **训练中**（2026-03-27 启动） |

**目的：** 单独验证 pred_bbox ROI 的收益，与实验二（GT bbox）做干净对比，每次只改一个变量。

**结果**

待测

---
**Stage 2 实验六 🏃 — pred_bbox ROI + Hard Mining（训练中）**

**exp_name:** `tumor_dynunet_predbbox_roi_hardmine`

| 项目 | 值 |
|------|----|
| init_ckpt | 实验二 best.pt（03-22-11-44-00）⚠️ 不等实验五，与实验五并行跑 |
| preprocessed_root | Task03_Liver_roi |
| Epochs | 300 |
| repeats | **6**（见下方说明） + small_tumor_thresh=500 + small_tumor_repeat_scale=**3** + no_tumor_repeat_scale=2 |
| random_margin | ✅ [8, **20**] |
| lr | 3e-4 |
| 预计时长 | **~50h** |
| 状态 | 🏃 **训练中**（2026-03-28 启动） |

**目的：** 在 pred_bbox 基础上叠加 hard mining，量化 hard mining 增益。与实验五（同 init_ckpt、同 pred_bbox、只差 hard mining）做干净对比。

⚠️ **实际参数与原计划的差异说明**

原计划 repeats=1、margin_max=24、small_tumor_repeat_scale=4，实际启动时沿用了实验五的命令模板，导致三处出入：
- `repeats=6`（与实验五一致，实测 ~5 min/epoch，加 hard mining 后约 10 min/epoch，300ep ≈ 50h，可接受）
- `margin_max=20`（与实验五一致）
- `small_tumor_repeat_scale=3`（原计划 4）

由于实验五也是 repeats=6，两者仍可做干净对比（hard mining 是唯一变量），消融结论不受影响。

**结果**

待测

---
<span style="color:#1a5276">**Stage 2 实验七 📋 — STU-Net 预训练权重迁移（准备中）**</span>

**exp_name:** `tumor_dynunet_stunet_init`

**背景：** STU-Net 是在大规模医学数据集上预训练的 nnUNet 风格模型，权重比从 Stage1 迁移更强。架构与我们的 DynUNet 高度一致，只需一处调整即可加载。

**架构对齐（当前 vs STU-Net-Small）**

| 参数 | 当前 DynUNet | STU-Net-Small | 需要改动 |
|------|------------|--------------|---------|
| resolution levels | 5（4次下采样） | 6（5次下采样） | ⚠️ 需加一层 |
| filters | [32,64,128,256,320] | [32,64,128,256,320,320] | ⚠️ 随架构自动对齐 |
| kernel_size | 全 3×3×3 | 全 3×3×3 | ✅ 一致 |
| instance norm + leaky relu | ✅ | ✅ | ✅ 一致 |

**代码改动（`medseg/models/dynunet.py`）**

```python
# 加第6层，与 STU-Net-Small 对齐
kernel_size=[[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
strides=[[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
upsample_kernel_size=[[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
```

96³ patch → 5次下采样后 bottleneck = 3³，可行。

**权重加载策略**

- 首尾层（in_channels / out_channels 不匹配）：随机初始化
- 中间所有层：直接拷贝 STU-Net 权重
- 加载方式：`strict=False`，打印命中率确认

**待办**

- [ ] 下载 STU-Net-Small 权重（HuggingFace: `uni-medical/STU-Net`）
- [ ] 改 `dynunet.py` 加第6层
- [ ] 写权重加载脚本 `scripts/load_stunet_weights.py`
- [ ] 新增 `--stunet_ckpt` 参数到 `train_tumor_roi.py`
- [ ] 用实验六同等设置训练，只换 init → 对比收益

| 项目 | 值 |
|------|----|
| init_ckpt | STU-Net-Small pretrained |
| preprocessed_root | Task03_Liver_roi |
| Epochs | 300 |
| repeats | 1 + hard mining（与实验六一致，只换初始化） |
| random_margin | ✅ [8, 24] |
| 预计时长 | **~50h** |
| 状态 | 📋 准备中（等实验五/六启动后开始代码准备） |

**结果**

待测

---
<span style="color:#7d6608">**论文对比: MedSAM2 bbox 推理 📋 — 零样本对比（待安排）**</span>

**目的：** 论文必做。用 MedSAM2 做零样本肿瘤分割，提供外部方法对比基线，不需要训练。

**推理流程**

```
Stage1 liver pred → 提取 bbox → 作为 MedSAM2 prompt → 推理 tumor mask → 评估 Dice
```

**配置**

| 项目 | 值 |
|------|----|
| 模型 | MedSAM2（SAM2 医学微调版） |
| prompt | Stage1 pred liver bbox（3D bounding box） |
| 数据 | test split，13 cases |
| 训练 | ❌ 零样本，不微调 |
| 评估 | 与实验二/六同口径（--tta --min_tumor_size 100） |
| 状态 | 📋 待安排（穿插在 GPU 等待期间完成） |

**结果**

待测

---
**横向对比**

| 实验 | 核心变化 | Val tumor Dice | Test tumor Dice | 状态 |
|------|---------|---------------|----------------|------|
| 实验一 | 基础版，82 阳性 case | 0.3801 | 0.5677 | ✅ |
| 实验二 | +92 case +jitter +margin | **0.4697** | 0.5816 | ✅ |
| 实验三 | hardfinetune，lr=3e-4 | 0.4603（中断） | 未测 | ❌ |
| 实验四 | pred_bbox ROI + hard mining，repeats=6 | — | — | ❌ 停止 |
| 实验五 | pred_bbox ROI only，repeats=6，300ep | 待测 | 待测 | 🏃 训练中 |
| 实验六 | pred_bbox ROI + hard mining，repeats=6，300ep | 待测 | 待测 | 🏃 训练中 |
| 实验七 | STU-Net 预训练权重迁移（6层架构对齐） | 待测 | 待测 | 📋 准备中 |
| eval: TTA | 实验二 ckpt + 8 种翻转（test） | — | **0.5965** ✅ 最优 | ✅ |
| eval: min_tumor_size=100（val） | 实验二 ckpt + TTA | **0.5757** | — | ✅ |
| eval: min_tumor_size 扫参（test） | 实验二 ckpt + TTA，size_100 最优 | — | **0.5950** | ✅ |
| 论文对比: MedSAM2 | bbox 推理，零样本对比 | — | 待测 | 📋 待安排 |

> Val Dice（训练时滑窗验证）和 Test Dice（完整两阶段推理）不直接可比。

---
<span style="color:#2c3e50">**nnUNet 官方基线（论文对比用）**</span>

来源: `/home/pumengyu/nnUNet_result/`，官方 nnUNet v1，**仅 1 fold（fold 0）**，验证集 **19 cases**（划分不同，不可 per-case 对比）。

| 指标 | Liver (Class 1) | Tumor (Class 2) |
|------|----------------|----------------|
| **Dice** | **0.9652 ± 0.011** | **0.7569 ± 0.215** |
| Jaccard | 0.9330 | 0.6444 |
| Precision | 0.9706 | 0.8797 |
| Recall | 0.9605 | 0.7133 |

**注意：** 1-fold 比 nnUNet 论文（5-fold ensemble）低约 3~5 点；5-fold 约 0.81~0.82。公平对比需在同一 test split（seed=0）上跑 nnUNet 推理。

---
<span style="color:#2c3e50">**下一步计划**</span>

**消融链（GPU 排队顺序）**

```
实验五（pred_bbox，训练中）
   ↓ 同时
实验六（pred_bbox + hard mining，GPU 空出立马启动）
   ↓ 五/六完成后
实验七（STU-Net 预训练权重迁移）
```

---

**GPU 调度**

| GPU | 当前任务 | 下一个任务 |
|-----|---------|----------|
| GPU 0 | 实验五 🏃 | 实验七（等实验五结束） |
| GPU 1 | 空闲待命 | **实验六（立马启动）** |

---

**中期 — 两通道输入（实验五/六完成后视结果决定）**

Stage 2 目前只输入 CT（1 通道）。增加空间先验通道，帮助模型定位肿瘤：

| 通道 | 来源 | 作用 |
|------|------|------|
| Ch1: CT | 原始 CT（肝脏 ROI 内） | 主要信号 |
| Ch2: 肝脏分割 | Stage 1 输出 liver mask | 提供肝脏边界，减少肝外误报（↓FDR） |

**方案 B — Stage 2 双通道**：直接修改 Stage 2 输入为 `[CT, liver_mask]`，主要压 FDR。

---

**论文必做 — MedSAM2 bbox 推理对比**

零样本推理，不训练，只需跑推理脚本得到对比数字，在空闲时穿插完成。

---

**可选消融 — SwinUNETR + SuPreM**

放消融实验章节，不作主力方向。视前几个实验结果决定是否做。

---
<span style="color:#2c3e50">**经验总结**</span>

**lr 选择（fine-tune from init_ckpt）**

- lr=1e-4 收敛太慢（epoch 5 时 tumor dice 仅 0.06，提前终止）
- 换了 loss/patch 相当于重新适应，需要足够学习率
- **建议范围：lr=5e-4 ~ 8e-4**（fine-tune 时）；若同 loss/patch 小幅调整才适合 1e-4

---

**代码改动记录 — transforms 增强（2026-03-25，commit 994e6ee）**

| 文件 | 改动 |
|------|------|
| `medseg/data/transforms_offline.py` | `num_samples: 1→2`；新增 `RandZoomd(min_zoom=0.7, max_zoom=1.4, prob=0.3)` |
| `twostage/dataset_tumor_roi.py` | 移除 `len(out)>1` 的 RuntimeError |
| `scripts/train_tumor_roi.py` | train DataLoader 加 `collate_fn=list_data_collate` |

---

**eval 脚本说明**

- eval 输出目录名取自 `stage2_ckpt` 的父目录名（与训练 run 一一对应）
- TTA：`--tta` 对 D/H/W 三轴 8 种翻转取平均，推理时间 ×8（commit 2014275）
- **后续 eval 固定参数：** `--tta --min_tumor_size 100`